# 25.5 — Neural theorem proving

**Lesson tagline.** Neural theorem proving keeps proof structure, but replaces exact symbol matching with soft unification.

Soft unification lets aliases and near matches support proof search, while proof scores remain products of fact confidence, unification, and rule confidence.

Save a copy to Drive to edit.

In [ ]:

import math
import itertools
import random
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

SEED = 25
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(z):
    z = np.asarray(z, dtype=float)
    return 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))


def binary_cross_entropy(p, y):
    p = np.clip(np.asarray(p, dtype=float), 1e-12, 1.0 - 1e-12)
    y = np.asarray(y, dtype=float)
    terms = -y * np.log(p) - (1.0 - y) * np.log(1.0 - p)
    return float(np.mean(terms))



def normalize(v):
    v = np.asarray(v, dtype=float)
    norm = np.linalg.norm(v)
    if norm == 0.0:
        return v
    return v / norm


def soft_unify(a, b, embeddings):
    return float(np.dot(normalize(embeddings[a]), normalize(embeddings[b])))


def path_score(fact_confidence, unification_score, rule_confidence):
    return float(fact_confidence * unification_score * rule_confidence)


def soft_or_paths(scores):
    scores = np.asarray(scores, dtype=float)
    return float(1.0 - np.prod(1.0 - scores))


def temperature_weights(similarities, tau):
    sims = np.asarray(similarities, dtype=float)
    shifted = sims / tau
    shifted = shifted - np.max(shifted)
    weights = np.exp(shifted)
    return weights / np.sum(weights)


def prove_query(query, facts, embeddings, rule_confidence=0.8, tau=0.5, type_gate=None, max_paths=50):
    source, target = query
    paths = []
    searched = 0
    for fact_source, fact_mid, fact_conf in facts:
        searched += 1
        if type_gate is not None and type_gate.get(fact_source) != type_gate.get(source):
            continue
        u = max(0.0, soft_unify(source, fact_source, embeddings))
        first = path_score(fact_conf, u, rule_confidence)
        for second_source, second_target, second_conf in facts:
            searched += 1
            if fact_mid != second_source:
                continue
            if second_target != target:
                continue
            score = first * second_conf
            paths.append((score, (fact_source, fact_mid, second_target)))
            if len(paths) >= max_paths:
                break
    scores = [score for score, path in paths]
    return soft_or_paths(scores) if scores else 0.0, paths, searched


def make_ntp_ladder():
    emb = {
        'alice': np.array([1.0, 0.0]),
        'bob': np.array([0.9701425001453318, 0.24253562503633294]),
        'carol': np.array([0.0, 1.0]),
    }
    rungs = []
    facts1 = [('alice', 'bob', 0.9), ('bob', 'carol', 0.8)]
    rungs.append({'name': 'D1 lesson three-symbol proof toy', 'embeddings': emb, 'facts': facts1, 'queries': [('alice', 'carol')]})
    emb2 = dict(emb)
    emb2.update({'alicia': np.array([0.99, 0.05]), 'dan': np.array([0.1, 0.99])})
    facts2 = facts1 + [('alicia', 'bob', 0.85), ('bob', 'dan', 0.75)]
    rungs.append({'name': 'D2 clean alias graph short proofs', 'embeddings': emb2, 'facts': facts2, 'queries': [('alice', 'dan'), ('alicia', 'carol')]})
    emb3 = dict(emb2)
    emb3.update({'mallory': np.array([0.90, 0.35]), 'erin': np.array([0.2, 0.98])})
    facts3 = facts2 + [('mallory', 'bob', 0.7), ('bob', 'erin', 0.6), ('alice', 'mallory', 0.4)]
    rungs.append({'name': 'D3 near-neighbor distractors multiple paths', 'embeddings': emb3, 'facts': facts3, 'queries': [('alice', 'erin'), ('mallory', 'carol')]})
    emb4 = dict(emb3)
    for i in range(6):
        angle = 0.25 * i
        emb4[f'entity{i}'] = np.array([math.cos(angle), math.sin(angle)])
    facts4 = facts3 + [(f'entity{i}', f'entity{i + 1}', 0.65 + 0.03 * i) for i in range(5)]
    rungs.append({'name': 'D4 inline synonym-like knowledge graph', 'embeddings': emb4, 'facts': facts4, 'queries': [('entity0', 'entity2'), ('alice', 'entity2')]})
    emb5 = dict(emb4)
    rng = np.random.default_rng(25)
    for i in range(12):
        emb5[f'node{i}'] = normalize(np.array([1.0, 0.08 * i]) + 0.05 * rng.normal(size=2))
    facts5 = facts4 + [(f'node{i}', f'node{i + 1}', 0.55 + 0.02 * (i % 4)) for i in range(11)]
    facts5 += [('node0', 'bob', 0.52), ('mallory', 'node5', 0.58), ('node8', 'carol', 0.57)]
    type_gate = {name: 'person' for name in emb5}
    type_gate['mallory'] = 'distractor'
    rungs.append({'name': 'D5 long proof accidental-neighbor stress', 'embeddings': emb5, 'facts': facts5, 'queries': [('alice', 'carol'), ('node0', 'carol')], 'type_gate': type_gate})
    return rungs


## The concept, built once (D1)

Soft unification is cosine similarity $u(a,b)=e_a^	op e_b$. A proof path has score $s_{path}=f\cdot u\cdot r$, and paths combine by $S=1-\prod_j(1-s_j)$.

In [ ]:

embeddings = {
    'alice': np.array([1.0, 0.0]),
    'bob': np.array([0.9701425001453318, 0.24253562503633294]),
    'carol': np.array([0.0, 1.0]),
}
cosines = [soft_unify('alice', name, embeddings) for name in ['alice', 'bob', 'carol']]
score = path_score(0.9, cosines[1], 0.8)
combined = soft_or_paths([0.6985026001046389, 0.25, 0.10])
print('cosines:', cosines)
print('path score:', score)
print('soft OR:', combined)
assert np.allclose(cosines, np.array([1.0, 0.9701425001453318, 0.0]))
assert abs(score - 0.6985026001046389) < 1e-15
assert abs(combined - 0.7964892550706313) < 1e-15


Temperature controls diffuse support. At $	au=1.0$, the weak symbol gets mass $0.15811219504644652$, while rule weight $w=1$ yields theorem score $0.9143615688372891$.

In [ ]:

weights_02 = temperature_weights(cosines, 0.2)
weights_05 = temperature_weights(cosines, 0.5)
weights_10 = temperature_weights(cosines, 1.0)
weighted_score = soft_or_paths([0.8731282501307987, 0.25, 0.10])
print('tau=0.2:', weights_02)
print('tau=0.5:', weights_05)
print('tau=1.0:', weights_10)
print('w=1 theorem:', weighted_score)
assert np.allclose(weights_02, np.array([0.5359210030511573, 0.4614037595663572, 0.002675237382485479]))
assert np.allclose(weights_05, np.array([0.4739434710247303, 0.44643847071117955, 0.0796180582640902]))
assert np.allclose(weights_10, np.array([0.4272237377129011, 0.4146640672406524, 0.15811219504644652]))
assert abs(weighted_score - 0.9143615688372891) < 1e-15


## The dataset ladder

The F16 ladder increases proof graph size, aliases, distractors, multiple paths, long paths, and accidental nearest neighbors.

In [ ]:

rungs = make_ntp_ladder()
for i, rung in enumerate(rungs, start=1):
    print(f"D{i}: {rung['name']} symbols={len(rung['embeddings'])} facts={len(rung['facts'])} queries={len(rung['queries'])}")
    print('sample facts:', rung['facts'][:3])


## Run the SAME method across D1–D5

In [ ]:

results = []
for i, rung in enumerate(rungs, start=1):
    scores = []
    searched_total = 0
    for query in rung['queries']:
        score, paths, searched = prove_query(query, rung['facts'], rung['embeddings'])
        scores.append(score)
        searched_total += searched
    solved_rate = float(np.mean(np.asarray(scores) > 0.25))
    results.append({'D': i, 'score': float(np.mean(scores)), 'solved': solved_rate, 'searched': searched_total, 'scores': scores})
print('rung | mean_score | solved_rate | nodes_or_paths_searched')
for row in results:
    print(f"D{row['D']} | {row['score']:.4f} | {row['solved']:.3f} | {row['searched']}")


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for col, rung in enumerate(rungs):
    coords = np.array([normalize(v) for v in rung['embeddings'].values()])
    names = list(rung['embeddings'].keys())
    axes[0, col].scatter(coords[:, 0], coords[:, 1], s=20)
    for idx, name in enumerate(names[:6]):
        axes[0, col].text(coords[idx, 0], coords[idx, 1], name, fontsize=7)
    axes[0, col].set_title(f"D{col + 1} symbol space")
axes[1, 0].plot([row['D'] for row in results], [row['solved'] for row in results], marker='o', label='solved rate')
axes[1, 0].plot([row['D'] for row in results], [row['searched'] for row in results], marker='s', label='searched')
axes[1, 0].set_title('proof metric vs complexity')
axes[1, 0].legend()
for ax in axes[1, 1:]:
    ax.axis('off')
plt.tight_layout()
plt.show()


## Pitfall on the hardest rung

D5 can prove through an accidental nearest neighbor. Lower temperature, type constraints, or exact-match gates make the proof search more cautious.

In [ ]:

d5 = rungs[-1]
loose_score, loose_paths, loose_searched = prove_query(('alice', 'carol'), d5['facts'], d5['embeddings'], rule_confidence=0.95)
gated_score, gated_paths, gated_searched = prove_query(('alice', 'carol'), d5['facts'], d5['embeddings'], rule_confidence=0.8, type_gate=d5['type_gate'])
print('loose proof score:', loose_score)
print('loose top paths:', sorted(loose_paths, reverse=True)[:5])
print('gated proof score:', gated_score)
print('gated top paths:', sorted(gated_paths, reverse=True)[:5])
print('searched loose/gated:', loose_searched, gated_searched)


## Evaluate it + Practice

- **Metric.** Track solved proof queries, calibration score, and paths searched, and compare it with an exact-match symbolic prover with no soft unification.
- **Cheap sanity check.** D1 must reproduce cosines, path score, temperature weights, and soft OR exactly.
- **Ablation.** set all embeddings to one-hot exact identities and compare alias proofs.
- **Failure signals.** high scores arise from weak nearest-neighbor chains rather than intended facts.

### Practice prompts


- Add a second rule confidence and combine paths.

- Experiment with temperature gates on D5.

- Create a false alias and measure calibration error.